**Generated By Gemini, Primpted, Edited and Debugged By Manim Community Nepal**

In [ ]:
from manim import *
import numpy as np

class Advanced3DPlaneGeometry(ThreeDScene):
    def construct(self):
        # --- CAMERA & SETUP ---
        self.set_camera_orientation(phi=65 * DEGREES, theta=-40 * DEGREES)
        
        # --- UI LAYOUT ---
        title = Title(r"Planes, Lines, Point & Distances").scale(0.8)
        self.add_fixed_in_frame_mobjects(title)
        
        # 3D Axes
        axes = ThreeDAxes(
            x_range=[-4, 4, 1],
            y_range=[-4, 4, 1],
            z_range=[-3, 3, 1],
            x_length=6,
            y_length=6,
            z_length=4
        ).shift(RIGHT * 2.5 + DOWN * 0.5)
        
        axis_labels = axes.get_axis_labels(x_label="x", y_label="y", z_label="z")
        
        self.play(Create(axes), Create(axis_labels))
        
        # --- SCENES EXECUTION ---
        self.scene_angle_planes(axes)
        self.scene_angle_line_plane(axes)
        self.scene_point_distance(axes)
        self.scene_parallel_distance(axes)
        self.scene_intersection(axes)
        
        self.wait(2)

    # --- HELPER: TEXT MANAGER ---
    def update_text(self, content_group):
        if hasattr(self, 'current_text'):
            self.play(FadeOut(self.current_text))
        
        content_group.scale(0.75).to_edge(LEFT, buff=0.5).shift(UP*0.5)
        self.add_fixed_in_frame_mobjects(content_group)
        self.play(Write(content_group))
        self.current_text = content_group

    # --- HELPER: PLANE CREATOR (MODIFIED) ---
    def get_plane(self, axes, normal=(0,0,1), d=0, color=BLUE, opacity=0.5, size=3, stroke_color=WHITE):
        """
        Added stroke_color parameter. 
        Default is WHITE (shows grid). Set to None to hide grid.
        """
        a, b, c = normal
        
        def func(u, v):
            x, y = u, v
            if abs(c) > 0.01:
                z = (-d - a*x - b*y) / c
            else:
                z = 0 
            return axes.c2p(x, y, z)

        plane = Surface(
            func,
            u_range=[-size, size],
            v_range=[-size, size],
            resolution=(8, 8),
            fill_color=color,
            fill_opacity=opacity,
            checkerboard_colors=[color, color],
            stroke_width=0.5,
            stroke_color=stroke_color # Uses the passed parameter
        )
        return plane

    # -------------------------------------------------------------------------
    # 1. ANGLE BETWEEN TWO PLANES (Grids ON)
    # -------------------------------------------------------------------------
    def scene_angle_planes(self, axes):
        t1 = Tex(r"\textbf{1. Angle Between Two Planes}", color=YELLOW, font_size=36)
        t2 = MathTex(r"P_1: a_1x + b_1y + c_1z + d_1 = 0", color=BLUE, font_size=32)
        t3 = MathTex(r"P_2: a_2x + b_2y + c_2z + d_2 = 0", color=GREEN, font_size=32)
        t4 = MathTex(r"\cos \theta = \frac{|\vec{n_1} \cdot \vec{n_2}|}{|\vec{n_1}| |\vec{n_2}|}", font_size=32)
        t5 = MathTex(r"\cos \theta = \frac{|a_1a_2 + b_1b_2 + c_1c_2|}{\sqrt{a_1^2+b_1^2+c_1^2}\sqrt{a_2^2+b_2^2+c_2^2}}", font_size=28)
        t6 = Tex(r"Perp ($\theta=90^\circ$): $a_1a_2 + b_1b_2 + c_1c_2 = 0$", color=RED, font_size=30)
        t7 = Tex(r"Parallel ($\theta=0^\circ$): $\frac{a_1}{a_2} = \frac{b_1}{b_2} = \frac{c_1}{c_2}$", color=ORANGE, font_size=30)
        group = VGroup(t1, t2, t3, t4, t5, t6, t7).arrange(DOWN, aligned_edge=LEFT, buff=0.2)
        self.update_text(group)

        plane1 = self.get_plane(axes, normal=(0,0,1), color=BLUE, opacity=0.4, stroke_color=WHITE)
        plane2 = self.get_plane(axes, normal=(0,1,1), color=GREEN, opacity=0.4, stroke_color=WHITE)
        
        n1_vec = np.array([0, 0, 2])
        n2_vec = np.array([0, 1.5, 1.5])
        arrow1 = Arrow3D(start=axes.c2p(0,0,0), end=axes.c2p(*n1_vec), color=BLUE_A)
        arrow2 = Arrow3D(start=axes.c2p(0,0,0), end=axes.c2p(*n2_vec), color=GREEN_A)
        lbl1 = MathTex(r"\vec{n_1}", color=BLUE_A).rotate(90*DEGREES, RIGHT).next_to(arrow1.get_end(), RIGHT)
        lbl2 = MathTex(r"\vec{n_2}", color=GREEN_A).rotate(90*DEGREES, RIGHT).next_to(arrow2.get_end(), UP)

        arc_pts = []
        for t in np.linspace(0, 1, 20):
            v1 = n1_vec / np.linalg.norm(n1_vec)
            v2 = n2_vec / np.linalg.norm(n2_vec)
            v_curr = (1-t)*v1 + t*v2
            v_curr = v_curr / np.linalg.norm(v_curr) * 0.8 
            arc_pts.append(axes.c2p(*v_curr))
        angle_curve = VMobject(color=YELLOW, stroke_width=4).set_points_smoothly(arc_pts)
        angle_lbl = MathTex(r"\theta", color=YELLOW).rotate(90*DEGREES, RIGHT).move_to(axes.c2p(0, 0.3, 0.6))

        self.play(Create(plane1), Create(plane2))
        self.play(GrowFromPoint(arrow1, axes.c2p(0,0,0)), GrowFromPoint(arrow2, axes.c2p(0,0,0)))
        self.play(Write(lbl1), Write(lbl2))
        self.play(Create(angle_curve), Write(angle_lbl))
        self.wait(3)
        self.play(FadeOut(Group(plane1, plane2, arrow1, arrow2, lbl1, lbl2, angle_curve, angle_lbl)))

    # -------------------------------------------------------------------------
    # 2. ANGLE: LINE AND PLANE (Grids ON)
    # -------------------------------------------------------------------------
    def scene_angle_line_plane(self, axes):
        t1 = Tex(r"\textbf{2. Angle: Line \& Plane}", color=YELLOW, font_size=36)
        t2 = MathTex(r"\text{Line: } \vec{r} = \vec{a} + \lambda \vec{b} \quad (\vec{b}=\langle l,m,n \rangle)", color=RED, font_size=32)
        t3 = MathTex(r"\text{Plane: } \vec{r} \cdot \vec{n} = d \quad (\vec{n}=\langle a,b,c \rangle)", color=BLUE, font_size=32)
        t4 = MathTex(r"\sin \phi = \frac{|\vec{b} \cdot \vec{n}|}{|\vec{b}| |\vec{n}|}", font_size=32)
        t5 = MathTex(r"\sin \phi = \frac{|al + bm + cn|}{\sqrt{a^2+b^2+c^2}\sqrt{l^2+m^2+n^2}}", font_size=28)
        t6 = Tex(r"If $\vec{b} \cdot \vec{n} = 0 \Rightarrow$ Line $\parallel$ Plane", color=ORANGE, font_size=30)
        t7 = Tex(r"If $\vec{b} \parallel \vec{n} \Rightarrow$ Line $\perp$ Plane", color=RED, font_size=30)
        group = VGroup(t1, t2, t3, t4, t5, t6, t7).arrange(DOWN, aligned_edge=LEFT, buff=0.2)
        self.update_text(group)

        plane = self.get_plane(axes, normal=(0,0,1), color=BLUE, opacity=0.3, stroke_color=WHITE)
        n_arrow = Arrow3D(start=axes.c2p(0,0,0), end=axes.c2p(0,0,2.5), color=BLUE_A)
        line_arrow = Arrow3D(start=axes.c2p(0,0,0), end=axes.c2p(0,3,3), color=RED)
        proj_arrow = Arrow3D(start=axes.c2p(0,0,0), end=axes.c2p(0,3,0), color=GREY)
        
        n_lbl = MathTex(r"\vec{n}", color=BLUE_A).rotate(90*DEGREES, RIGHT).next_to(n_arrow.get_end(), LEFT)
        line_lbl = MathTex(r"\vec{b}", color=RED).rotate(90*DEGREES, RIGHT).next_to(line_arrow.get_end(), RIGHT)
        proj_lbl = Tex("proj", color=GREY, font_size=20).rotate(90*DEGREES, RIGHT).next_to(proj_arrow.get_end(), DOWN)

        phi_pts = [axes.c2p(0, 1.5*np.cos(t), 1.5*np.sin(t)) for t in np.linspace(0, 45*DEGREES, 20)]
        phi_arc = VMobject(color=YELLOW, stroke_width=4).set_points_smoothly(phi_pts)
        phi_txt = MathTex(r"\phi", color=YELLOW).rotate(90*DEGREES, RIGHT).move_to(axes.c2p(0, 1.8, 0.6))

        theta_pts = [axes.c2p(0, 1.0*np.cos(t), 1.0*np.sin(t)) for t in np.linspace(45*DEGREES, 90*DEGREES, 20)]
        theta_arc = VMobject(color=WHITE, stroke_width=2).set_points_smoothly(theta_pts)
        theta_txt = MathTex(r"\theta", color=WHITE, font_size=24).rotate(90*DEGREES, RIGHT).move_to(axes.c2p(0, 0.4, 1.2))

        self.play(Create(plane))
        self.play(GrowFromPoint(n_arrow, axes.c2p(0,0,0)), Write(n_lbl))
        self.play(GrowFromPoint(line_arrow, axes.c2p(0,0,0)), Write(line_lbl))
        self.play(GrowFromPoint(proj_arrow, axes.c2p(0,0,0)), Write(proj_lbl))
        self.play(Create(phi_arc), Write(phi_txt))
        self.play(Create(theta_arc), Write(theta_txt))
        self.wait(3)
        self.play(FadeOut(Group(plane, n_arrow, n_lbl, line_arrow, line_lbl, proj_arrow, proj_lbl, phi_arc, phi_txt, theta_arc, theta_txt)))

    # -------------------------------------------------------------------------
    # 3. DISTANCE: POINT TO PLANE (Grids ON)
    # -------------------------------------------------------------------------
    def scene_point_distance(self, axes):
        t1 = Tex(r"\textbf{3. Distance: Point to Plane}", color=YELLOW, font_size=36)
        t2 = MathTex(r"P(x_1, y_1, z_1)", color=RED, font_size=32)
        t3 = MathTex(r"\text{Plane: } ax+by+cz+d=0", color=BLUE, font_size=32)
        t4 = MathTex(r"D", r"=", r"\frac{|\vec{AP} \cdot \vec{n}|}{|\vec{n}|}", font_size=32)
        t4[0].set_color(YELLOW)
        t4[2][1:4].set_color(TEAL)
        t5 = Tex(r"where ", r"$A$", r" is any point on plane", font_size=30)
        t5[1].set_color(GREEN)
        t6 = MathTex(r"D", r"=", r"\frac{|ax_1 + by_1 + cz_1 + d|}{\sqrt{a^2+b^2+c^2}}", font_size=32)
        t6[0].set_color(YELLOW)
        group = VGroup(t1, t2, t3, t4, t5, t6).arrange(DOWN, aligned_edge=LEFT, buff=0.2)
        self.update_text(group)

        plane = self.get_plane(axes, normal=(0,0,1), d=1, color=BLUE, stroke_color=WHITE) 
        p_loc = axes.c2p(1, 1, 2)
        dot_p = Dot3D(point=p_loc, color=RED, radius=0.15)
        dot_p_lbl = MathTex(r"P", color=RED).rotate(90*DEGREES, RIGHT).next_to(dot_p, UP)
        a_loc = axes.c2p(-1.5, -1, -1)
        dot_a = Dot3D(point=a_loc, color=GREEN, radius=0.15)
        dot_a_lbl = MathTex(r"A", color=GREEN).rotate(90*DEGREES, RIGHT).next_to(dot_a, LEFT)
        vec_ap = Arrow3D(start=a_loc, end=p_loc, color=TEAL)
        vec_ap_lbl = MathTex(r"\vec{AP}", color=TEAL).rotate(90*DEGREES, RIGHT).move_to(axes.c2p(-0.2, 0, 0.5))
        proj_loc = axes.c2p(1, 1, -1)
        dist_line = DashedLine(start=p_loc, end=proj_loc, color=YELLOW, stroke_width=4)
        dist_lbl = MathTex("D", color=YELLOW).rotate(90*DEGREES, RIGHT).next_to(dist_line, RIGHT)
        ra_1 = Line3D(start=axes.c2p(1, 1, -0.7), end=axes.c2p(1.3, 1, -0.7), color=WHITE)
        ra_2 = Line3D(start=axes.c2p(1.3, 1, -0.7), end=axes.c2p(1.3, 1, -1), color=WHITE)

        self.play(Create(plane))
        self.play(FadeIn(dot_a), Write(dot_a_lbl))
        self.play(FadeIn(dot_p), Write(dot_p_lbl))
        self.play(GrowFromPoint(vec_ap, point=a_loc), Write(vec_ap_lbl))
        self.play(Create(dist_line), Write(dist_lbl))
        self.play(Create(ra_1), Create(ra_2))
        self.wait(3)
        self.play(FadeOut(Group(plane, dot_p, dot_p_lbl, dot_a, dot_a_lbl, vec_ap, vec_ap_lbl, dist_line, dist_lbl, ra_1, ra_2)))

    # -------------------------------------------------------------------------
    # 4. DISTANCE: PARALLEL PLANES (Grids ON)
    # -------------------------------------------------------------------------
    def scene_parallel_distance(self, axes):
        t1 = Tex(r"\textbf{4. Distance: Parallel Planes}", color=YELLOW, font_size=36)
        t2 = MathTex(r"P_1: \vec{r} \cdot \vec{n} = d_1 \quad (ax+by+cz+d_1=0)", color=BLUE, font_size=32)
        t3 = MathTex(r"P_2: \vec{r} \cdot \vec{n} = d_2 \quad (ax+by+cz+d_2=0)", color=GREEN, font_size=32)
        t4 = MathTex(r"D = \frac{|d_1 - d_2|}{|\vec{n}|} = \frac{|d_1 - d_2|}{\sqrt{a^2+b^2+c^2}}", font_size=32)
        group = VGroup(t1, t2, t3, t4).arrange(DOWN, aligned_edge=LEFT, buff=0.3)
        self.update_text(group)

        plane1 = self.get_plane(axes, normal=(0,0,1), d=-1, color=BLUE, opacity=0.4, stroke_color=WHITE)
        plane2 = self.get_plane(axes, normal=(0,0,1), d=1, color=GREEN, opacity=0.4, stroke_color=WHITE)
        start = axes.c2p(0, 0, 1)
        end = axes.c2p(0, 0, -1)
        line = DashedLine(start=start, end=end, color=YELLOW, stroke_width=4)
        d_text = MathTex("D", color=YELLOW).rotate(90*DEGREES, RIGHT).next_to(line, RIGHT)
        
        self.play(Create(plane1), Create(plane2))
        self.play(Create(line), Write(d_text))
        self.wait(3)
        self.play(FadeOut(Group(plane1, plane2, line, d_text)))

    # -------------------------------------------------------------------------
    # 5. FAMILY OF PLANES (Bisector NO Grid, others Grid ON)
    # -------------------------------------------------------------------------
    def scene_intersection(self, axes):
        t1 = Tex(r"\textbf{5. Family of Planes}", color=YELLOW, font_size=40)
        t2 = MathTex(r"P_1 + k P_2 = 0", color=RED, font_size=40)
        t3 = MathTex(r"(a_1x+b_1y+c_1z+d_1) + k(a_2x+b_2y+c_2z+d_2) = 0", font_size=32)
        t4 = Tex(r"Passes through line of intersection of $P_1$ and $P_2$", font_size=30)
        group = VGroup(t1, t2, t3, t4).arrange(DOWN, aligned_edge=LEFT, buff=0.3)
        self.update_text(group)

        # Base planes P1 (Blue) and P2 (Green) - Keep Grids
        p1 = self.get_plane(axes, normal=(0,1,0), color=BLUE, opacity=0.5, stroke_color=WHITE)
        p2 = self.get_plane(axes, normal=(0,0,1), color=GREEN, opacity=0.5, stroke_color=WHITE)
        p1_lbl = MathTex("P_1", color=BLUE, font_size=48).rotate(90*DEGREES, RIGHT).move_to(axes.c2p(2, 0.2, 2))
        p2_lbl = MathTex("P_2", color=GREEN, font_size=48).rotate(90*DEGREES, RIGHT).move_to(axes.c2p(2, 2, 0.2))
        int_line = Line3D(start=axes.c2p(-4,0,0), end=axes.c2p(4,0,0), color=WHITE, thickness=3)
        
        # Static Bisecting Plane (Red) - REMOVE GRID (stroke_color=None)
        bisecting_plane = self.get_plane(axes, normal=(0,1,1), color=RED, opacity=0.6, stroke_color=None)
        bisector_lbl = MathTex(r"\text{Bisector } (k=1)", color=RED, font_size=40).rotate(90*DEGREES, RIGHT).move_to(axes.c2p(0, 1.5, 1.5))

        self.play(Create(p1), Create(p2), Write(p1_lbl), Write(p2_lbl))
        self.play(Create(int_line))
        self.play(Create(bisecting_plane), Write(bisector_lbl))
        self.wait(4)
        self.play(FadeOut(Group(p1, p2, p1_lbl, p2_lbl, int_line, bisecting_plane, bisector_lbl)))


%manim -ql -v warning Advanced3DPlaneGeometry

Manim Community v0.19.0